<div align="center">

# Sherlock, the Robot Detective
A crime solving Production Engine

Group Assignment - Knowledge Representation & Reasoning

Fall, 2026
</div>


![](images/Sherlock.png)

Introducing our production engine, Sherlock, the Robot Detective, here investigating his own footprints.


# Task 2 - Reasoning through time

## Sherlock, a robot of many talents

Sherlock, the Robot Detective, is a robot of many talents. Apart form being able to investigate crimes and solve the mystery using forward and backward chaining, he also has the ability to move through time, and perform some actions, such as gather evidence!

Sherlock has one free hand (he never puts his magnifying glass down), so he can hold one object.

In [ ]:
from reasoning.domains import  TASK2_INITIAL_FACTS, TASK_2_SCHEMAS
from reasoning.engine import WorkingMemory, step, choose, instances

from visualiser.world_view import WorldView
view = WorldView(TASK2_INITIAL_FACTS)

# Setting Sherlock's working memory
wm = WorkingMemory(TASK2_INITIAL_FACTS)

## Setting the scene

![](images/Ravenwick_Manor.png)

The crime takes place at the **Ravenwick Manor**, where there are 8 rooms;
- the library,
- the study,
- the foyer,
- the gallery,
- the laboratory,
- the hallway,
- the garden,
- the terrace

Here we can see the visual representation of Sherlocks current working memory:

In [ ]:
view.show(wm, 0);

## The dissapearing mansion (§2.1.1)

When Sherlock operates without switching on his persistance, he will lose his memory after he moves.

In [ ]:
# Separate world for demonstrating the failure without persistence.
wm_no_persistence = WorkingMemory(TASK2_INITIAL_FACTS)


In [ ]:
# Moving from the terrace to the garden
move_action = choose( wm_no_persistence, ("move", "terrace", "garden"), TASK_2_SCHEMAS, 0)

# Moving with persistance switched off
step(wm_no_persistence, move_action, 0, persistence=False)

view.show(wm_no_persistence, 1);

With no persistence all the rooms dissapear, all paths between rooms, all artifacts. Recording only an action’s effects loses knowledge about everything else at the next time step.

## Mission: Bring the evidence to the foyer room, leaving the other items undisturbed.

The original working memory is still intact and contains all the evidence. So let's start again with Sherlock's main memory.

The below process needs to be run in the order it appears, as each part of the process assumes Sherlock is in the correct location.

In [ ]:
t = 0

view.show(wm, t);

Sherlocks mission it to move the letter opener (in the study), a sample vial in the laboratory, and a letter in the library.

The key should remain where it is, on the terrace.

In [ ]:
def perform(plan, verbose=False):
    global t

    for action_name in plan:
        action = choose(wm, action_name, TASK_2_SCHEMAS, t)
        step(wm, action, t, persistence=True)
        t += 1

        if(verbose):
            print(f"\nStep {t}: {action_name}")
            for fact in sorted(wm.facts):
                if fact[-1] == t:
                    print(fact)

        assert ("at", "key", "terrace", t) in wm.facts, (
            "Persistence failed: the key's location was lost."
        )
    view.show(wm, t)
        

### Sherlock moves to the foyer

Sherlock enters the mansion.

When he moves around, now with persistence, the rooms and objects all remain.

In [ ]:
plan = [("move", "terrace", "garden"),("move", "garden", "foyer")]
perform(plan)

### Collect the letter opener from the study

To be able to pick up the letter opener, Sherlock needs to first move to the study and then grab the letter opener.

In [ ]:
plan= [("move", "foyer", "hallway"), ("move", "hallway", "gallery"), ("move", "gallery", "study"), ("grab", "letter_opener", "study")]
perform(plan)

As shown, Sherlock now holds the letter opener. Let's see if he can carry that to the foyer.

In [ ]:


perform([("move", "study", "gallery"),("move", "gallery", "hallway"),("move", "hallway", "foyer")])


And there it is! Now Sherlock releases it so he can collect the rest of the evidence.

In [ ]:

perform([("release", "letter_opener", "foyer")])

Sherlock now is empty handed and the letter opener is successfully moved to the Foyer.

### Collect the letter from the library

Sherlock now retrieves the letter. The letter opener must remain in the foyer while Sherlock works elsewhere; its successful delivery must persist.

In [ ]:
plan = [
    ("move", "foyer", "hallway"),
    ("move", "hallway", "library"),
    ("grab", "letter", "library"),
    ("move", "library", "hallway"),
    ("move", "hallway", "foyer"),
    ("release", "letter", "foyer")]
perform(plan)

Two done one to go!

### Collect the sample vial from the laboratory
Finally, Sherlock retrieves the sample vial. Both previously delivered objects must remain in the foyer.

The same generic move, grab, and release schemas work for every object and location through variable bindings.

In [ ]:
perform([
    ("move", "foyer", "hallway"),
    ("move", "hallway", "gallery"),
    ("move", "gallery", "laboratory"),
    ("grab", "sample_vial", "laboratory"),
    ("move", "laboratory", "gallery"),
    ("move", "gallery", "hallway"),
    ("move", "hallway", "foyer"),
    ("release", "sample_vial", "foyer"),
])

Well done Sherlock! 

This was done in 24 steps.

### Verification

Just to be safe, lets verify the results:

In [ ]:
expected = {
    ("at", "letter_opener", "foyer", t),
    ("at", "letter", "foyer", t),
    ("at", "sample_vial", "foyer", t),
    ("at", "sherlock", "foyer", t),
    ("empty", "sherlock", t),
    ("at", "key", "terrace", t),
}

missing = expected - wm.facts
assert not missing, f"Missing final facts: {missing}"

# Delivered objects should have no remaining holding facts
# or alternative locations at the current time step.
for obj in ("letter_opener", "letter", "sample_vial"):
    assert ("holding", obj, t) not in wm.facts

    locations = {
        fact[2]
        for fact in wm.facts
        if len(fact) == 4
        and fact[0] == "at"
        and fact[1] == obj
        and fact[-1] == t
    }
    assert locations == {"foyer"}, (obj, locations)

print(f"Mission complete at step {t}: all evidence is in the foyer.")
print("The key remained on the terrace throughout.")